# Module 6: Minor League Discipline and MLB Translation

The central finding of this case study. Both Volpe and Dominguez had **elite plate discipline in the minors**. Volpe's chase rate was 15%. Dominguez's was 17.8%. These are elite numbers by any standard.

Against MLB pitching, both regressed:
- Volpe: 15% → 30.6% chase rate (+15.6 points)
- Dominguez: 17.8% → 31.1% chase rate (+13.3 points)

The issue isn't talent. It isn't discipline. **It's calibration** — their pitch recognition was trained on minor league stuff, and MLB breaking balls and offspeed are a fundamentally different challenge. The development pipeline did not appear to bridge that gap before the call-up.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="urllib3")

import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from fire_fishman.data.statcast import get_statcast_pitches, get_statcast_batter
from fire_fishman.features.pitch_level import (
    compute_plate_discipline, compute_whiff_by_pitch_type,
    compute_velo_tier_performance
)

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 7)

## 1. Pull MiLB Statcast Data

MiLB Statcast coverage is limited — we have Spring Training and select minor league games. Small samples, but the signal is clear.

In [ ]:
# Pull pre-debut pitch data (cached locally as parquet after the first pull)
volpe_milb = get_statcast_batter(683011, "2021-04-01", "2023-03-25")
dom_milb = get_statcast_batter(691176, "2022-04-01", "2023-08-31")

# MLB data
pitches_2023 = get_statcast_pitches(2023)
pitches_2024 = get_statcast_pitches(2024)
mlb_pitches = pd.concat([pitches_2023, pitches_2024])

print(f"Volpe MiLB pitches: {len(volpe_milb)}")
print(f"Dominguez MiLB pitches: {len(dom_milb)}")

## 2. Side-by-Side: MiLB vs MLB Discipline

In [ ]:
def bootstrap_ci(data, stat_fn, n_boot=2000, ci=0.95, seed=42):
    """Compute bootstrap confidence interval for a statistic."""
    rng = np.random.RandomState(seed)
    boot_stats = []
    for _ in range(n_boot):
        sample = data.sample(n=len(data), replace=True, random_state=rng)
        boot_stats.append(stat_fn(sample))
    alpha = (1 - ci) / 2
    return np.percentile(boot_stats, [100*alpha, 100*(1-alpha)])


milb_mlb_compare = {}
for name, pid, milb_data in [("Anthony Volpe", 683011, volpe_milb),
                              ("Jasson Dominguez", 691176, dom_milb)]:
    print(f"\n{'='*70}")
    print(f"  {name.upper()}: MiLB → MLB  (MiLB n={len(milb_data)} pitches)")
    print(f"{'='*70}")
    
    milb_disc = compute_plate_discipline(milb_data, pid)
    mlb_disc = compute_plate_discipline(mlb_pitches, pid)
    milb_whiff = compute_whiff_by_pitch_type(milb_data, pid)
    mlb_whiff = compute_whiff_by_pitch_type(mlb_pitches, pid)
    
    print(f"\n  {'Metric':<30} {'MiLB':>10} {'95% CI':>18} {'MLB':>10} {'Change':>10}")
    print(f"  {'-'*80}")
    
    # Prepare MiLB batter pitches for bootstrap
    milb_bp = milb_data[milb_data['batter'] == pid].copy()
    from fire_fishman.features.pitch_level import _vectorized_zone_swing_whiff
    milb_bp = _vectorized_zone_swing_whiff(milb_bp)
    
    for metric in ['chase_rate', 'whiff_rate', 'zone_contact_rate']:
        mv = milb_disc.get(metric, np.nan)
        mlv = mlb_disc.get(metric, np.nan)
        diff = mlv - mv if not (np.isnan(mv) or np.isnan(mlv)) else np.nan
        flag = ' !!!' if isinstance(diff, float) and abs(diff) > 0.10 else ''
        
        # Bootstrap CI for MiLB metric
        if metric == 'chase_rate':
            oz = milb_bp[~milb_bp['in_zone'] & milb_bp['plate_x'].notna()]
            if len(oz) >= 10:
                ci_lo, ci_hi = bootstrap_ci(oz, lambda s: s['is_swing'].mean())
                ci_str = f'[{ci_lo:.1%}, {ci_hi:.1%}]'
            else:
                ci_str = '[n too small]'
        elif metric == 'whiff_rate':
            sw = milb_bp[milb_bp['is_swing']]
            if len(sw) >= 10:
                ci_lo, ci_hi = bootstrap_ci(sw, lambda s: s['is_whiff'].mean())
                ci_str = f'[{ci_lo:.1%}, {ci_hi:.1%}]'
            else:
                ci_str = '[n too small]'
        elif metric == 'zone_contact_rate':
            iz_sw = milb_bp[milb_bp['in_zone'] & milb_bp['is_swing']]
            if len(iz_sw) >= 10:
                ci_lo, ci_hi = bootstrap_ci(iz_sw, lambda s: 1 - s['is_whiff'].mean())
                ci_str = f'[{ci_lo:.1%}, {ci_hi:.1%}]'
            else:
                ci_str = '[n too small]'
        else:
            ci_str = ''
        
        print(f"  {metric:<30} {mv:>9.1%} {ci_str:>18} {mlv:>9.1%} {diff:>+9.1%}{flag}")
    
    for metric in ['whiff_rate_fastball', 'whiff_rate_breaking', 'whiff_rate_offspeed',
                   'chase_rate_breaking', 'chase_rate_offspeed']:
        mv = milb_whiff.get(metric, np.nan)
        mlv = mlb_whiff.get(metric, np.nan)
        if np.isnan(mv):
            continue
        diff = mlv - mv
        flag = ' !!!' if abs(diff) > 0.10 else ''
        print(f"  {metric:<30} {mv:>9.1%} {'':>18} {mlv:>9.1%} {diff:>+9.1%}{flag}")
    
    print(f"\n  Note: MiLB sample is {len(milb_data)} pitches. CIs show the uncertainty")
    print(f"  in point estimates — directional signal is strong but precision is limited.")

    # Collect the headline metrics for the comparison figure in the next cell
    fourth_label, fourth_col = (
        ("Whiff Rate\nBreaking", "whiff_rate_breaking") if name == "Anthony Volpe"
        else ("Whiff Rate\nFastball", "whiff_rate_fastball")
    )
    milb_mlb_compare[name.split()[-1]] = {
        "Chase Rate": (milb_disc.get("chase_rate", np.nan), mlb_disc.get("chase_rate", np.nan)),
        "Chase Rate\nBreaking": (milb_whiff.get("chase_rate_breaking", np.nan),
                                  mlb_whiff.get("chase_rate_breaking", np.nan)),
        "Chase Rate\nOffspeed": (milb_whiff.get("chase_rate_offspeed", np.nan),
                                  mlb_whiff.get("chase_rate_offspeed", np.nan)),
        fourth_label: (milb_whiff.get(fourth_col, np.nan), mlb_whiff.get(fourth_col, np.nan)),
    }


In [ ]:
# Visualization: MiLB vs MLB collapse — values computed in the cell above
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (name, metrics) in zip(axes, milb_mlb_compare.items()):
    labels = list(metrics.keys())
    milb_vals = [v[0] for v in metrics.values()]
    mlb_vals = [v[1] for v in metrics.values()]
    
    x = np.arange(len(labels))
    width = 0.35
    
    ax.bar(x - width/2, milb_vals, width, label="MiLB (Pre-Debut)", color="#2ecc71", alpha=0.85)
    ax.bar(x + width/2, mlb_vals, width, label="MLB", color="#e74c3c", alpha=0.85)
    
    for i, (mv, mlv) in enumerate(zip(milb_vals, mlb_vals)):
        diff = mlv - mv
        color = "#e74c3c" if diff > 0 else "#2ecc71"
        ax.annotate(f"{diff:+.0%}", (i + width/2, mlv), fontsize=11, fontweight="bold",
                   ha="center", va="bottom", color=color)
    
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel("Rate", fontsize=12)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    ax.set_ylim(0, 0.50)
    ax.set_title(f"{name}\nMiLB → MLB Discipline Collapse", fontsize=13, fontweight="bold")
    ax.legend(fontsize=10)

plt.suptitle("Minor League Discipline and MLB Translation", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../outputs/figures/milb_vs_mlb.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. What's Different About MLB Pitching?

It's not just that pitches are faster. The key differences:

1. **Breaking ball quality** — MLB sliders have more sweep, tighter spin, and later break. MLB pitchers have sharper break, higher velocity, and better tunneling than minor league pitchers — the same pitch recognition skills that worked in AAA may not transfer.

2. **Tunneling** — MLB pitchers tunnel their fastball and breaking ball far better. The pitches look identical at the decision point. In the minors, hitters can identify pitch type earlier.

3. **Sequencing** — MLB staffs have data-driven pitch sequencing. They know Dominguez chases offspeed and they feed it to him. In the minors, sequencing is more random.

4. **Velocity stacking** — MLB pitchers establish 96+ early in counts, then throw breaking balls off it. The speed differential is larger and the hitter's timing is more disrupted.

## 4. The Development Failure

Both prospects had the **skill** of discipline. What they didn't have was the **calibration** — their pitch recognition system was trained on minor league stuff.

This is a fixable problem, but it has to be fixed **before** the call-up:

- High-speed pitch machines that replicate MLB-quality movement
- VR-based pitch recognition training with actual MLB pitch tunnels
- AAA pitching coaches instructed to throw MLB-style sequences against top prospects
- Statcast-based readiness gates: don't promote until chase rate holds against MLB-quality stuff

The Yankees chose to let their prospects calibrate on the job, in front of 45,000 people, with their confidence on the line. That's a development pipeline failure.

### Data Limitations
The MiLB data here is from Spring Training / select minor league games (89-254 pitches). Full minor league Statcast is expanding but not yet comprehensive. The directional signal is strong — both prospects showed dramatically better discipline pre-MLB — but the exact numbers should be treated as indicative, not definitive.